In [1]:
# （1）构建 MCP 客户端
# 对官方 mcp SDK 的最小封装：一个类支持三种传输方式
#   stdio           本地子进程（自己写的 Server）
#   streamable_http 远程 HTTP 长连接（公开可直连的 Server）
#   sse             远程 SSE（老一点的 Server）
# 连接过程固定三步：打开传输 -> 建 ClientSession -> initialize 握手
# 握手之后就能 list_tools（发现工具）和 call_tool（调用工具）

import json
import sys
from contextlib import AsyncExitStack, asynccontextmanager
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.sse import sse_client
from mcp.client.stdio import stdio_client
from mcp.client.streamable_http import streamablehttp_client

# 公开可直连的远程天气 MCP 服务端：免注册、免 API Key，数据来自 Open-Meteo
# 工具：weather_now / air_quality_now / nearby_sensors，城市名要用英文（Beijing）
REMOTE_WEATHER_URL = "https://modelmarket.dev/tools/mcp/weather"


class MCPClient:
    """一个 MCP 连接 = 一条传输通道 + 一个 ClientSession。"""

    def __init__(self, name, transport_factory, errlog_path=None):
        self.name = name
        self._transport_factory = transport_factory  # 无参函数，返回传输的异步上下文管理器
        self._stack = None
        self.session = None
        self.server_info = None  # 握手后拿到的服务端信息（name / version）
        self.errlog_path = errlog_path  # stdio 模式下 Server 的 stderr 日志落在哪

    # ---- 三种传输的构造方法，返回的都是同一个 MCPClient ----

    @classmethod
    def stdio(cls, name, command, args=(), env=None, cwd=None, errlog_path=None):
        """本地 Server：把 Server 当子进程拉起来，用标准输入输出通信。"""
        params = StdioServerParameters(command=command, args=list(args), env=env, cwd=cwd)

        # ⚠️ Jupyter 专用处理（见本 cell 底部注释）：cell 里的 sys.stderr 没有 fileno()，
        # 不能直接当子进程的 stderr，必须写进一个真实文件。
        log_path = Path(errlog_path) if errlog_path else None
        if log_path is None and args:
            log_path = Path(args[0]).with_suffix(".stderr.log")
        if log_path is None:  # 推不出日志路径就退回默认行为（仅 .py 环境可用）
            return cls(name, lambda: stdio_client(params), errlog_path)

        @asynccontextmanager
        async def transport():
            with open(log_path, "a", encoding="utf-8", errors="replace") as errlog:
                async with stdio_client(params, errlog=errlog) as streams:
                    yield streams

        return cls(name, transport, log_path)

    @classmethod
    def http(cls, name, url, headers=None, timeout=60):
        """远程 Server：streamable-http 传输。"""
        return cls(name, lambda: streamablehttp_client(url, headers=headers, timeout=timeout))

    @classmethod
    def sse(cls, name, url, headers=None, timeout=30):
        """远程 Server：SSE 传输。"""
        return cls(name, lambda: sse_client(url, headers=headers, timeout=timeout))

    # ---- 上下文管理：进入时握手，退出时自动断开 ----

    async def __aenter__(self):
        self._stack = AsyncExitStack()
        # streamable_http 会多返回一个 session_id 回调，用 *_ 忽略掉
        read_stream, write_stream, *_ = await self._stack.enter_async_context(self._transport_factory())
        self.session = await self._stack.enter_async_context(ClientSession(read_stream, write_stream))
        result = await self.session.initialize()
        self.server_info = result.serverInfo
        return self

    async def __aexit__(self, exc_type, exc, tb):
        await self._stack.aclose()

    # ---- 工具发现与调用 ----

    async def list_tools(self):
        """tools/list：返回服务端暴露的工具对象列表（名字、描述、参数 Schema）。"""
        return (await self.session.list_tools()).tools

    async def call_tool(self, name, arguments=None):
        """tools/call：返回原始 CallToolResult（content 是「内容块列表」）。"""
        return await self.session.call_tool(name, arguments or {})

    async def call_text(self, name, arguments=None):
        """调用工具并只取文本，把 content 块列表拼成一个字符串。"""
        result = await self.call_tool(name, arguments)
        text = "\n".join(b.text for b in result.content if getattr(b, "text", None) is not None)
        if result.isError:
            raise RuntimeError(f"工具 {name} 返回错误: {text}")
        return text

    async def call_json(self, name, arguments=None):
        """调用工具并把返回文本按 JSON 解析（天气 Server 基本都返回 JSON 字符串）。"""
        return json.loads(await self.call_text(name, arguments))


print("✅ MCPClient 已定义（stdio / streamable_http / sse 三种传输通用）")

# ─────────────────────────────────────────────────────────────────────────────
# 【.py 文件里能跑、ipynb cell 里不能跑的写法，注释保留】
#
# ① 切换控制台编码。在 .py 里这样写没问题：
#      for _stream in (sys.stdout, sys.stderr):
#          _stream.reconfigure(encoding="utf-8", errors="replace")
#    但 Jupyter 的 sys.stdout/sys.stderr 是 OutStream，没有 reconfigure 方法，
#    直接调会 AttributeError: 'OutStream' object has no attribute 'reconfigure'。
#    （Jupyter 本身已按 UTF-8 输出，cell 里不需要也不能这么做，故上面整段省略。）
#
# ② stdio 客户端的默认 errlog。在 .py 里可以直接用默认的 errlog=sys.stderr：
#      async with stdio_client(params) as (read, write):        # errlog 默认 sys.stderr
#          async with ClientSession(read, write) as session:
#              await session.initialize()
#    但 cell 里 sys.stderr 是 OutStream，拿不到真实 fileno()，subprocess 启动子进程时
#    会抛 io.UnsupportedOperation: fileno。所以上面 MCPClient.stdio 改成把 Server 的
#    stderr 重定向到一个真实文件（*.stderr.log），再用它当 errlog。
# ─────────────────────────────────────────────────────────────────────────────

✅ MCPClient 已定义（stdio / streamable_http / sse 三种传输通用）


In [2]:
# （2）在本地构建天气 MCP 服务端，用（1）的客户端连接它并获取当天的北京天气
# 先把服务端源码写成独立文件（Server 必须是独立进程），再以子进程 + stdio 方式拉起

SERVER_DIR = Path.cwd() if Path("demo-mcp.ipynb").exists() else Path.cwd() / "demo"
SERVER_PATH = (SERVER_DIR / "weather_mcp_server.py").resolve()

SERVER_CODE = r'''"""
自建天气 MCP 服务端（本地，stdio 传输）

- FastMCP：官方高层封装，@mcp.tool() 把普通函数注册成 MCP 工具，
  类型注解 -> 参数 Schema，docstring -> 工具描述
- 和 LangChain @tool 几乎一一对应，区别是这里跑在独立进程，跨进程 JSON-RPC 调用
- ⚠️ stdio 模式下 stdout 被协议报文独占，任何 print() 都会破坏 JSON-RPC，日志一律写 stderr

手动运行可验证（阻塞等 stdin，Ctrl+C 退出）：python weather_mcp_server.py
"""

import json
import logging
import sys
from datetime import date

from mcp.server.fastmcp import FastMCP

# Windows 下 stdout 默认 GBK，中文会写坏协议报文，必须切 UTF-8
for _stream in (sys.stdout, sys.stderr):
    if hasattr(_stream, "reconfigure"):
        _stream.reconfigure(encoding="utf-8", errors="replace")

logging.basicConfig(level=logging.INFO, stream=sys.stderr, format="[weather-server] %(message)s")
logger = logging.getLogger("weather-server")

# 1. 创建 Server（name 会在握手时告诉客户端）
mcp = FastMCP("local-weather-server")

# 当天天气的 Mock 数据（本地演示不联网，真实数据见第（3）块）
MOCK_WEATHER = {
    "北京": {"天气": "晴", "温度": "25°C", "湿度": "40%", "风力": "微风"},
    "上海": {"天气": "多云", "温度": "22°C", "湿度": "65%", "风力": "东南风3级"},
    "广州": {"天气": "小雨", "温度": "28°C", "湿度": "80%", "风力": "南风2级"},
    "深圳": {"天气": "阴", "温度": "26°C", "湿度": "70%", "风力": "东风2级"},
}
UNKNOWN = {"天气": "未知", "温度": "N/A", "湿度": "N/A", "风力": "N/A"}


# 2. 注册工具
@mcp.tool()
def get_weather(city: str) -> str:
    """查询指定城市当天的天气信息。

    Args:
        city: 城市名称，如：北京、上海、广州、深圳
    """
    data = MOCK_WEATHER.get(city, UNKNOWN)
    logger.info("get_weather called: city=%s", city)
    return json.dumps(
        {"城市": city, "日期": date.today().isoformat(), "数据源": "local-mock", **data},
        ensure_ascii=False,
    )


@mcp.tool()
def list_supported_cities() -> str:
    """列出当前服务支持查询的所有城市。"""
    return json.dumps({"支持城市": list(MOCK_WEATHER)}, ensure_ascii=False)


# 3. 以 stdio 方式对外服务
if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

SERVER_PATH.write_text(SERVER_CODE, encoding="utf-8")
print(f"已生成本地服务端: {SERVER_PATH}\n")


async def block2():
    # 用当前 kernel 的解释器启动 Server 子进程，走 stdio 传输
    async with MCPClient.stdio(
        name="local-weather",
        command=sys.executable,
        args=[str(SERVER_PATH)],
    ) as client:
        print(f"已连接: {client.server_info.name} v{client.server_info.version}")

        # tools/list：动态发现服务端有哪些工具
        tools = await client.list_tools()
        print(f"\n发现 {len(tools)} 个工具:")
        for t in tools:
            print(f"  - {t.name:<22} {(t.description or '').strip().splitlines()[0]}")

        # tools/call：获取当天北京天气（不经过 LLM，看 MCP 的原始返回）
        print("\n【当天北京天气】")
        print(await client.call_text("get_weather", {"city": "北京"}))

        data = await client.call_json("get_weather", {"city": "北京"})
        print(f"\n解析成 dict: {data}")
        print(f"支持的城市: {await client.call_json('list_supported_cities')}")

        # Server 的 stderr 日志（logging 只能写 stderr，stdout 是协议通道）
        errlog = Path(client.errlog_path)
        if errlog.exists():
            tail = errlog.read_text(encoding="utf-8", errors="replace").strip().splitlines()[-5:]
            print(f"\n📄 Server 端日志 {errlog.name}:\n  " + "\n  ".join(tail))


# cell 里用顶层 await（.py 里要写 asyncio.run，见底部注释）
await block2()

# ─────────────────────────────────────────────────────────────────────────────
# 【.py 文件里能跑、ipynb cell 里不能跑的写法，注释保留】
#
# ① 服务端最后一句 mcp.run(transport="stdio")：
#      if __name__ == "__main__":
#          mcp.run(transport="stdio")
#    在 .py 里能正常起 stdio 服务；但直接粘进 cell 会失败——mcp.run 内部又要
#    asyncio.run，而 cell 已在运行事件循环，报 RuntimeError: Already running
#    asyncio in this thread。所以服务端必须写成独立 .py 文件、当子进程启动，
#    不能在 cell 里直接 run（上面正是这么做的）。
#
# ② 启动入口。在 .py 里通常这样跑异步主函数：
#      if __name__ == "__main__":
#          asyncio.run(block2())
#    但 asyncio.run() 在 cell 里会报错（cell 已有运行中的事件循环）：
#      RuntimeError: asyncio.run() cannot be called from a running event loop
#    所以 cell 里直接用顶层 await：`await block2()`（见上面）。
# ─────────────────────────────────────────────────────────────────────────────

已生成本地服务端: E:\Programs\Python\Python Code\ai-agent-writer-dev\demo\weather_mcp_server.py



已连接: local-weather-server v1.30.0

发现 2 个工具:
  - get_weather            查询指定城市当天的天气信息。
  - list_supported_cities  列出当前服务支持查询的所有城市。

【当天北京天气】
{"城市": "北京", "日期": "2026-10-20", "数据源": "local-mock", "天气": "晴", "温度": "25°C", "湿度": "40%", "风力": "微风"}

解析成 dict: {'城市': '北京', '日期': '2026-10-20', '数据源': 'local-mock', '天气': '晴', '温度': '25°C', '湿度': '40%', '风力': '微风'}
支持的城市: {'支持城市': ['北京', '上海', '广州', '深圳']}

📄 Server 端日志 weather_mcp_server.stderr.log:
  [weather-server] Processing request of type CallToolRequest
  [weather-server] get_weather called: city=北京
  [weather-server] Processing request of type CallToolRequest
  [weather-server] get_weather called: city=北京
  [weather-server] Processing request of type CallToolRequest


In [3]:
# （3）用（1）的客户端连接公开可直连的【远程】天气 MCP 服务端，获取当天的北京天气
# 远程服务端 https://modelmarket.dev/tools/mcp/weather ：streamable-http，免注册免 Key
# 数据源是 Open-Meteo 实时观测；城市名必须用英文（中文会被 GeoNames 拒绝），
# 所以同时演示 latitude/longitude 兜底（坐标一定可用）
# 注意：API 和本地 stdio 完全一样，只是把 MCPClient.stdio 换成 MCPClient.http

BEIJING_LATLON = {"latitude": 39.9042, "longitude": 116.4074}


async def block3():
    async with MCPClient.http(name="remote-weather", url=REMOTE_WEATHER_URL) as client:
        print(f"已连接远程服务端: {client.server_info.name} v{client.server_info.version}")

        tools = await client.list_tools()
        print(f"\n远程发现 {len(tools)} 个工具:")
        for t in tools:
            print(f"  - {t.name:<20} {(t.description or '').strip().splitlines()[0][:70]}")

        # tools/call：当天北京实时天气
        print("\n【当天北京天气】weather_now(city='Beijing')")
        result = await client.call_json("weather_now", {"city": "Beijing"})

        # 返回的 JSON 层级较深，只把有用字段摘出来
        reading = result["result"]["reading"]
        values = reading["values"]
        resolved = result["result"]["resolved"]
        print(f"  观测时间 : {reading['ts']}")
        print(f"  匹配地点 : {resolved['matched_place']} (relay {reading['device_id']})")
        print(f"  温度     : {values['temperature_c']} °C")
        print(f"  湿度     : {values['humidity_pct']} %")
        print(f"  气压     : {values['pressure_hpa']} hPa")
        print(f"  风速     : {values['wind_mps']} m/s")

        # 同样的查询改用经纬度
        print("\n【按经纬度再查一次】")
        by_ll = await client.call_json("weather_now", BEIJING_LATLON)
        v = by_ll["result"]["reading"]["values"]
        print(f"  {v['temperature_c']}°C / 湿度 {v['humidity_pct']}% / 风速 {v['wind_mps']}m/s")

        # 顺带查当天空气质量
        print("\n【当天北京空气质量】")
        air = await client.call_json("air_quality_now", {"city": "Beijing"})
        print(f"  {air['result']['reading']['values']}")


# cell 里用顶层 await（.py 里要写 asyncio.run，见底部注释）
await block3()

# ─────────────────────────────────────────────────────────────────────────────
# 【.py 文件里能跑、ipynb cell 里不能跑的写法，注释保留】
#   if __name__ == "__main__":
#       asyncio.run(block3())          # .py 入口
# asyncio.run() 在 cell 里会报 RuntimeError: cannot be called from a running
# event loop，所以 cell 里改成顶层 await block3()（见上面）。
# ─────────────────────────────────────────────────────────────────────────────

已连接远程服务端: modelmarket-weather v1.0.0



远程发现 3 个工具:
  - weather_now          Current weather at a place: temperature (°C), humidity (%), pressure (
  - air_quality_now      Current air quality at a place: PM2.5 and PM10 (µg/m³), US and Europea
  - nearby_sensors       The nearest live public sensors to a point, one per layer asked for (w

【当天北京天气】weather_now(city='Beijing')


  观测时间 : 2026-10-20T14:00:15Z
  匹配地点 : Beijing (relay om-wx-beijing)
  温度     : 7.8 °C
  湿度     : 45 %
  气压     : 1019.8 hPa
  风速     : 1.8 m/s

【按经纬度再查一次】


  7.8°C / 湿度 45% / 风速 1.8m/s

【当天北京空气质量】


  {'pm2_5_ugm3': 22.7, 'pm10_ugm3': 34.5, 'us_aqi': 169, 'european_aqi': 114}


In [4]:
# （4）仿照 demo-langchain.py，通过 Agent 连接远程天气 MCP 服务端获取天气
# 和 demo-langchain.py 唯一的区别是 tools 的来源：
#   demo-langchain.py : 本地 @tool get_weather        -> 同进程函数调用
#   这里              : MultiServerMCPClient.get_tools() -> 远程 MCP 服务端，跨网络调用
# create_agent(model=llm, tools=tools) 本身一模一样

import os
from configparser import ConfigParser

from langchain.agents import create_agent
from langchain_core.messages import ToolMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

# 1. 读取项目根目录的 config.ini（和 demo-langchain.py 一致）
# 注意：cell 里没有 __file__，所以用 Path.cwd() 往上找，而不是 .py 里的 __file__ 定位
def find_config_ini(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "config.ini").exists():
            return p / "config.ini"
    raise FileNotFoundError("找不到 config.ini")


config = ConfigParser()
config.read(find_config_ini(), encoding="utf-8")

api_key = os.environ.get("LLM_API_KEY")
if not api_key:
    raise ValueError("环境变量 LLM_API_KEY 未设置，请先设置 API Key")

# 2. 创建 LLM
llm = ChatOpenAI(
    api_key=api_key,
    base_url=config.get("LLM", "api_url"),
    model=config.get("LLM", "model_name"),
    temperature=config.getfloat("LLM", "temperature"),
    max_tokens=1024,
)

# 3. MCP 连接配置：远程服务端用 streamable_http 传输（换成 stdio 就是本地 Server）
MCP_CONNECTIONS = {
    "remote-weather": {
        "transport": "streamable_http",
        "url": REMOTE_WEATHER_URL,
        "timeout": 60,
    }
}

SYSTEM_PROMPT = (
    "你是一个天气助手，可以调用工具查询实时天气与空气质量。"
    "注意：调用天气工具时城市名必须用英文（如 Beijing、Shanghai）。"
    "只能依据工具返回的真实数据作答，不要编造；关键指标用中文清晰汇总。"
)


def extract_text(content) -> str:
    """MCP 返回的是「内容块列表」，统一抽成可读字符串。"""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(
            b.get("text", str(b)) if isinstance(b, dict) else str(b) for b in content
        )
    return str(content)


def print_trace(messages, max_len=300):
    """打印一轮完整的 Agent-Loop 轨迹：LLM 决策 -> 工具调用 -> 工具返回 -> 最终回答。"""
    for msg in messages:
        for tc in getattr(msg, "tool_calls", None) or []:
            print(f"  🔧 [LLM 决定调用] {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)})")
        if isinstance(msg, ToolMessage):
            print(f"  📦 [工具返回] {msg.name} -> {extract_text(msg.content)[:max_len]}")
        elif msg.type == "ai" and msg.content:
            print(f"\n🤖 [Agent 回答]\n{msg.content}")


# 4. 创建 Agent：工具来自 MCP 动态发现，而不是本地 @tool 函数
mcp_client = MultiServerMCPClient(MCP_CONNECTIONS)
tools = await mcp_client.get_tools()
print(f"已从远程 MCP 服务端发现 {len(tools)} 个工具: {[t.name for t in tools]}")

agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)
chat_history = []


async def ask(question, recursion_limit=10):
    """提问一次并打印轨迹；chat_history 支持多轮对话（多次调用 ask 即可）。"""
    global chat_history
    print(f"👤 [用户] {question}\n" + "-" * 70)
    response = await agent.ainvoke(
        {"messages": chat_history + [("user", question)]},
        config={"recursion_limit": recursion_limit},  # 控制 Agent-Loop 最大迭代次数
    )
    chat_history = response["messages"]
    print_trace(response["messages"])
    return response


# cell 里直接问一个固定问题（用顶层 await，不用 input()，原因见底部注释）
_ = await ask("北京今天天气怎么样？适合出门跑步吗？")

# ─────────────────────────────────────────────────────────────────────────────
# 【.py 文件里能跑、ipynb cell 里不能跑的写法，注释保留】
#
# ① 交互式多轮循环。demo-langchain.py 里这段在 .py 中能正常跑：
#      while True:
#          user_input = input("用户: ").strip()      # ← cell 里会 StdinNotImplementedError
#          if user_input.lower() in ("exit", "quit"):
#              break
#          await ask(user_input)
#    但 ipynb cell 不支持 input()：
#      StdinNotImplementedError: raw_input was called, but this frontend does not
#      support input requests.
#    所以 cell 里改成写死一个问题、直接 await ask(...)；要真正的交互循环请用 .py 脚本。
#
# ② 启动入口。在 .py 里：
#      if __name__ == "__main__":
#          asyncio.run(main())
#    cell 里 asyncio.run 会报 RuntimeError（已有运行中的事件循环），故直接用顶层 await。
#
# ③ 定位配置文件。在 .py 里可以用 __file__：
#      BASE_PATH = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
#    但 cell 里没有 __file__（NameError: name '__file__' is not defined），
#    所以上面改用 Path.cwd() 往上找 config.ini。
# ─────────────────────────────────────────────────────────────────────────────

已从远程 MCP 服务端发现 3 个工具: ['weather_now', 'air_quality_now', 'nearby_sensors']
👤 [用户] 北京今天天气怎么样？适合出门跑步吗？
----------------------------------------------------------------------


  🔧 [LLM 决定调用] weather_now({"city": "Beijing"})
  🔧 [LLM 决定调用] air_quality_now({"city": "Beijing"})
  📦 [工具返回] weather_now -> {
 "capability_id": "gaia.weather.read@v1",
 "result": {
  "reading": {
   "device_id": "om-wx-beijing",
   "model": "GAIA-WS1 (Open-Meteo relay)",
   "site": "live-om-beijing",
   "firmware": "1.0.0",
   "seq": 440,
   "ts": "2026-10-20T14:00:28Z",
   "values": {
    "temperature_c": 7.8,
    "humi
  📦 [工具返回] air_quality_now -> {
 "capability_id": "gaia.air.read@v1",
 "result": {
  "reading": {
   "device_id": "om-aq-beijing",
   "model": "GAIA-AQ1 (Open-Meteo AQ relay)",
   "site": "live-om-beijing",
   "firmware": "1.0.0",
   "seq": 430,
   "ts": "2026-10-20T14:00:28Z",
   "values": {
    "pm2_5_ugm3": 22.7,
    "pm10_ug

🤖 [Agent 回答]
根据最新实时数据，北京今日天气与空气质量如下：

### 🌤 天气状况（2026-10-20 14:00 UTC）：
- **气温**：7.8 °C（体感偏凉）  
- **湿度**：45%（较舒适，不干燥也不闷热）  
- **气压**：1019.8 hPa（正常）  
- **风速**：1.8 m/s（微风，无明显阻力）

### 🌫 空气质量（同时间）：
- **PM2.5**：22.7 µg/m³ → **优-良水平**（WHO 24h指导值为